# Product & Growth Intelligence Platform

## 01 — Data Understanding

### Objective

This notebook establishes the analytical foundation for the Product & Growth Intelligence Platform using the RetailRocket e-commerce behavioral dataset.

The objective of this stage is to understand the structure, scale, and behavioral meaning of the available event-level data before performing data quality validation, feature engineering, funnel analysis, retention analysis, segmentation, and product intelligence.

### Key Questions

This notebook answers:

1. What does one event record represent?
2. What information is available about visitors, products, events, and transactions?
3. What behavioral event types are present?
4. How large is the dataset?
5. How many unique visitors and transactions are represented?
6. What is the temporal coverage of the data?
7. Are the relationships between visitors, products, events, and transactions suitable for downstream analysis?

### Analytical Principle

The project distinguishes carefully between different units of analysis:

- **Event** — one recorded visitor-product interaction
- **Visitor** — one unique `visitorid`
- **Product** — one unique `itemid`
- **Transaction** — one unique `transactionid`

These units are not interchangeable and will be kept separate throughout the analysis.

### Core Behavioral Journey

The primary behavioral journey represented by the dataset is:

**Product View → Add to Cart → Transaction**

The purpose of the project is not merely to count these events, but to use them to identify behavioral patterns, funnel friction, retention opportunities, and product/category growth opportunities supported by evidence.

## 1. Dataset Context

### RetailRocket E-commerce Behavioral Dataset

RetailRocket provides event-level behavioral data from an e-commerce environment.

The primary table used in this notebook is `events.csv`. It records visitor interactions with products through three core event types:

| Event | Behavioral Meaning |
|---|---|
| `view` | A visitor viewed a product |
| `addtocart` | A visitor added a product to their cart |
| `transaction` | A transaction occurred for a product |

The main event table contains the following fields:

| Column | Description |
|---|---|
| `timestamp` | Time at which the event occurred, stored as a Unix timestamp in milliseconds |
| `visitorid` | Identifier of the visitor |
| `event` | Type of behavioral event |
| `itemid` | Identifier of the product involved |
| `transactionid` | Identifier of the transaction when applicable |

The `events.csv` table is the primary behavioral foundation for the Product & Growth Intelligence Platform.

In [1]:
import pandas as pd
events = pd.read_csv( r"D:\Data science portfolio\03_Product_Growth_Intelligence\data\raw\events.csv")
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


## 2. Dataset Structure

Before analyzing behavioral patterns, we first establish the scale and technical structure of the event table.

This helps determine:

- how many behavioral records are available
- how many fields are available
- how Pandas has interpreted each field
- which columns may require transformation before analysis

In [2]:
events.shape

(2756101, 5)

In [3]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          object 
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB


## 3. Event Landscape

The `event` field defines the type of behavioral interaction recorded for each event.

Understanding the available event types establishes the behavioral funnel that will support downstream product and growth analysis.

The expected journey is:

**View → Add to Cart → Transaction**

We first verify the distinct event types present in the actual dataset before quantifying their frequency.

In [4]:
events["event"].unique()

array(['view', 'addtocart', 'transaction'], dtype=object)

In [5]:
event_counts = events["event"].value_counts()

event_counts

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

### Interpretation

The dataset contains three behavioral event types:

- `view` — product interest or browsing activity
- `addtocart` — stronger purchase intent
- `transaction` — recorded transaction activity

The event distribution is highly concentrated in product views, followed by a much smaller number of cart additions and transactions.

This establishes the central behavioral funnel for the project:

**View → Add to Cart → Transaction**

At this stage, these are event counts rather than conversion rates. Conversion metrics will be defined later using an explicit analytical unit and denominator rather than treating raw event proportions as conversion rates.

## 4. Visitor & Transaction Scale

Event counts describe the volume of recorded interactions, but they do not represent the number of individual visitors or transactions.

A visitor can generate multiple events, and a transaction can be represented by one or more transaction event records.

We therefore quantify visitors and transactions separately to maintain a clear distinction between:

- event-level activity
- visitor-level activity
- transaction-level activity

In [6]:
unique_visitors = events["visitorid"].nunique()
unique_transactions = events["transactionid"].nunique()

unique_visitors, unique_transactions

(1407580, 17672)

In [7]:
avg_events_per_visitor = len(events) / unique_visitors

avg_events_per_visitor

1.9580421716705267

### Interpretation

The event table contains **2,756,101 recorded events** generated by **1,407,580 unique visitors**.

This results in approximately **1.96 recorded events per visitor on average**.

The distinction between event volume and visitor volume is important for downstream analysis. Event-level counts measure interaction volume, while visitor-level metrics measure the number of distinct users exhibiting a behavior.

The dataset also contains **17,672 unique transaction IDs**, which is lower than the **22,457 transaction event records**. Therefore, transaction event rows and unique transactions must not be treated as the same metric.

## 5. Temporal Coverage & Datetime Preparation

The raw `timestamp` field is stored as Unix time in milliseconds. Before performing time-based behavioral analysis, we convert it into a Pandas datetime field.

This section establishes:

- the temporal coverage of the dataset
- the validity of the timestamp range
- an analysis-ready datetime field for downstream time-based analysis

In [8]:
events["datetime"] = pd.to_datetime( events["timestamp"], unit="ms")
events[["timestamp", "datetime"]].head()

,timestamp,datetime
0,1433221332117,2015-06-02 05:02:12.117
1,1433224214164,2015-06-02 05:50:14.164
2,1433221999827,2015-06-02 05:13:19.827
3,1433221955914,2015-06-02 05:12:35.914
4,1433221337106,2015-06-02 05:02:17.106


In [9]:
start_datetime = events["datetime"].min()
end_datetime = events["datetime"].max()

start_datetime, end_datetime

(Timestamp('2015-05-03 03:00:04.384000'),
 Timestamp('2015-09-18 02:59:47.788000'))

In [10]:
observation_days = (end_datetime - start_datetime).total_seconds() / (24 * 60 * 60)
observation_days

137.99980791666667

### Interpretation

The event data spans from **3 May 2015 to 18 September 2015**, providing approximately **138 days of observed behavioral activity**.

The raw Unix timestamps have been converted into a dedicated `datetime` field while preserving the original `timestamp` column.

This datetime field will be reused for downstream time-based analysis, including behavioral trends, cohort construction, retention measurement, recency calculations, and funnel analysis over time.

The observation period is based on the minimum and maximum timestamps in the event table rather than assuming that the first or last physical row represents the beginning or end of the dataset.

## 6. Initial Data Integrity

Before moving into detailed data-quality processing, we perform a small set of structural integrity checks on the event table.

The purpose is to identify issues that could distort downstream visitor, product, funnel, and transaction analysis.

The checks focus on:

- missing values in core identifiers
- exact duplicate event records
- validity of event categories
- consistency of transaction identifiers with transaction events

In [11]:
events.isna().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
datetime               0
dtype: int64

In [12]:
events.duplicated().sum()

np.int64(460)

### Initial Integrity Findings

The core fields `timestamp`, `visitorid`, `event`, `itemid`, and the derived `datetime` field contain no missing values.

The `transactionid` field contains missing values because transaction identifiers are only applicable to transaction events. This will be validated against event types before any cleaning decision is made.

The dataset contains **460 exact duplicate rows**. These records are flagged for further investigation rather than being removed automatically. Exact duplication may affect event counts and downstream behavioral metrics if the records represent unintended duplicate observations.

In [13]:
duplicate_count = events.duplicated().sum()
duplicate_percentage = (duplicate_count / len(events)) * 100

duplicate_count, duplicate_percentage

(np.int64(460), np.float64(0.016690244660845156))

In [14]:
duplicate_rows = events[events.duplicated(keep=False)].sort_values(
    ["timestamp", "visitorid", "itemid", "event"]
)

duplicate_rows.head(10)

,timestamp,visitorid,event,itemid,transactionid,datetime
1466550,1430639017006,771203,view,250578,NaN,2015-05-03 07:43:37.006
1471736,1430639017006,771203,view,250578,NaN,2015-05-03 07:43:37.006
1482986,1430780975099,234788,addtocart,33,NaN,2015-05-04 23:09:35.099
1496938,1430780975099,234788,addtocart,33,NaN,2015-05-04 23:09:35.099
1489030,1430788228572,594432,addtocart,311596,NaN,2015-05-05 01:10:28.572
1495975,1430788228572,594432,addtocart,311596,NaN,2015-05-05 01:10:28.572
1513411,1430851280377,882313,addtocart,168717,NaN,2015-05-05 18:41:20.377
1519889,1430851280377,882313,addtocart,168717,NaN,2015-05-05 18:41:20.377
1518734,1430863559288,734488,addtocart,351585,NaN,2015-05-05 22:05:59.288
1522559,1430863559288,734488,addtocart,351585,NaN,2015-05-05 22:05:59.288


### Duplicate Treatment Decision

The dataset contains **460 exact duplicate event records**, representing approximately **0.017%** of all records.

Inspection confirms that the identified duplicates are identical across the available event attributes, including timestamp, visitor, event type, product, transaction identifier, and derived datetime.

Because these records are indistinguishable duplicate observations, retaining them could artificially inflate behavioral event counts and downstream funnel metrics.

The raw `events` table is preserved unchanged. Exact duplicates are removed only when creating the analytical dataset used for downstream analysis.

In [15]:
events_clean = events.drop_duplicates().copy()

events_clean.shape

(2755641, 6)

In [16]:
events_clean.duplicated().sum()

np.int64(0)

### Transaction ID Integrity

The `transactionid` field is only expected to be populated for transaction events.

We therefore validate the relationship between `event` type and the presence of a transaction identifier before making any missing-value treatment decision.

In [17]:
transaction_id_by_event = (events_clean.groupby("event")["transactionid"].apply(lambda x: x.notna().sum()))
transaction_id_by_event

event
addtocart          0
transaction    22457
view               0
Name: transactionid, dtype: int64

### Interpretation

The validation confirms that transaction identifiers are present for `transaction` events and absent for `view` and `addtocart` events.

Therefore, the missing `transactionid` values are structurally expected rather than missing data errors.

No imputation or deletion of these missing transaction identifiers is required.

### Transaction Event Validation

Although transaction identifiers are present for transaction events, we perform one final consistency check to confirm that no transaction event is missing its identifier.

In [18]:
transaction_events_missing_id = events_clean[
    (events_clean["event"] == "transaction") &
    (events_clean["transactionid"].isna())
].shape[0]

transaction_events_missing_id

0

# 7. Key Findings & Data Understanding Summary

The initial analysis establishes the structure, scale, temporal coverage, and key integrity characteristics of the RetailRocket event dataset.

### Dataset Scale

- **2,756,101** raw event records
- **5** original event fields
- **1,407,580** unique visitors
- **17,672** unique transaction identifiers
- Approximately **138 days** of observed activity

### Behavioral Events

The dataset contains three event types:

- `view` — product browsing activity
- `addtocart` — stronger purchase intent
- `transaction` — recorded transaction activity

The event distribution establishes the central behavioral funnel:

**View → Add to Cart → Transaction**

### Data Integrity Findings

- Core identifiers contain no missing values.
- Missing `transactionid` values occur only for non-transaction events and are therefore structurally expected.
- All transaction events have a transaction identifier.
- **460 exact duplicate records** were identified.
- These duplicates represent approximately **0.017%** of the raw dataset and were removed from the analytical dataset.
- The raw `events` dataframe remains unchanged.
- The resulting `events_clean` dataset contains **2,755,641 records** and no exact duplicate rows.

### Analytical Implication

The cleaned event table provides a validated foundation for downstream behavioral analysis.

The next stage will transform event-level records into analytical structures required for:

- visitor behavior analysis
- funnel measurement
- cohort and retention analysis
- behavioral segmentation
- product and category intelligence
- growth opportunity identification

---

## Notebook Conclusion

This notebook established what the raw event data represents, validated its basic structural integrity, and documented the decisions required before downstream analysis.

No behavioral metrics, conversion rates, retention measures, or segments are calculated in this notebook. Those analyses will be developed from the validated analytical dataset in subsequent stages.